# 02 — Data Cleaning

**Objective:** Apply the findings from `01_data_understanding.ipynb` to produce a clean, training-ready dataset, using `src/preprocessing/data_cleaner.py` (FR-DATA-003/004/005). Per Handbook Section 10.11, the raw dataset is never modified — a cleaned copy is saved separately to `data/processed/`.

**Cleaning plan, based on real findings (not assumptions):**
1. Drop the 2 fully-empty junk columns found in notebook 01.
2. Re-confirm duplicate rows (already found to be zero, but re-checked formally here).
3. Re-confirm missing values in real columns (already found to be zero).
4. Run outlier detection (IQR + Z-score) and report — not auto-remove, per Handbook D.11/D.22.
5. Save the result to `data/processed/delhi_aqi_cleaned.csv`.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.preprocessing.data_loader import load_dataset
from src.preprocessing.data_cleaner import (
    remove_duplicate_rows, handle_missing_values,
    detect_outliers_iqr, detect_outliers_zscore,
)
from config.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR
from config.constants import REQUIRED_POLLUTANT_COLUMNS

df = load_dataset(RAW_DATA_DIR / "Delhi_AQI_Dataset.csv", date_format="%d/%m/%y")
print(f"Loaded: {df.shape}")

2026-08-18 10:03:40,041 | INFO | src.preprocessing.data_loader | Loaded dataset 'Delhi_AQI_Dataset.csv': 2191 rows, 11 columns.


Loaded: (2191, 11)


## Step 1 — drop fully-empty junk columns

Found in notebook 01: two trailing columns from a malformed header (`...,O3,,`). Dropping only columns confirmed 100% empty — not a blind `dropna(axis=1)`, so this can't accidentally remove a real column with a few missing values.

In [2]:
fully_empty_cols = [col for col in df.columns if df[col].isna().all()]
print(f"Dropping fully-empty columns: {fully_empty_cols}")

df = df.drop(columns=fully_empty_cols)
print(f"Shape after dropping junk columns: {df.shape}")
assert df.shape[1] == 9, "Expected exactly the 9 real columns to remain"

Dropping fully-empty columns: ['Unnamed: 9', 'Unnamed: 10']
Shape after dropping junk columns: (2191, 9)


## Step 2 — duplicate rows (FR-DATA-004)

In [3]:
df, n_removed = remove_duplicate_rows(df)
print(f"Duplicate rows removed: {n_removed}")
print(f"Shape: {df.shape}")

2026-08-18 10:03:40,076 | INFO | src.preprocessing.data_cleaner | remove_duplicate_rows: removed 0 duplicate row(s).


Duplicate rows removed: 0
Shape: (2191, 9)


**Result:** 0 duplicates removed, confirming notebook 01's finding. No action needed here beyond the formal check.

## Step 3 — missing values in real columns (FR-DATA-003)

In [4]:
missing_counts = df.isna().sum()
print(missing_counts)
print(f"\nTotal missing values across all real columns: {missing_counts.sum()}")

City     0
Date     0
AQI      0
PM2.5    0
PM10     0
NO2      0
SO2      0
CO       0
O3       0
dtype: int64

Total missing values across all real columns: 0


**Result:** zero missing values. `handle_missing_values()` exists and is unit-tested for when a future dataset (or an updated version of this one) does have gaps, but there is nothing to impute here — applying an imputation strategy to a column with 0 missing values would be a no-op, so it's correctly skipped rather than run pointlessly.

## Step 4 — outlier detection (FR-DATA-005, informational only)

In [5]:
cols_to_check = ["AQI"] + list(REQUIRED_POLLUTANT_COLUMNS)

iqr_results = detect_outliers_iqr(df, columns=cols_to_check)
zscore_results = detect_outliers_zscore(df, columns=cols_to_check, threshold=3.0)

print(f"{'Column':<8} {'IQR outliers':<14} {'Z-score outliers':<18}")
for col in cols_to_check:
    print(f"{col:<8} {iqr_results[col].count:<14} {zscore_results[col].count:<18}")

2026-08-18 10:03:40,115 | INFO | src.preprocessing.data_cleaner | detect_outliers_iqr: {'AQI': 0, 'PM2.5': 0, 'PM10': 0, 'NO2': 0, 'SO2': 0, 'CO': 0, 'O3': 0}
2026-08-18 10:03:40,119 | INFO | src.preprocessing.data_cleaner | detect_outliers_zscore: {'AQI': 0, 'PM2.5': 0, 'PM10': 0, 'NO2': 0, 'SO2': 0, 'CO': 0, 'O3': 0}


Column   IQR outliers   Z-score outliers  
AQI      0              0                 
PM2.5    0              0                 
PM10     0              0                 
NO2      0              0                 
SO2      0              0                 
CO       0              0                 
O3       0              0                 


**Interpretation:** zero outliers detected by either method, on every column. Combined with notebook 01's finding that pollutant columns are exact linear transforms of `AQI` (`PM2.5 = 0.55 × AQI`, confirmed by inspection), this makes sense: a value can only be an "outlier" relative to the shape of its own distribution, and AQI itself — while having a wide range (41–494) — doesn't have extreme tail values in this file. **Decision: no outlier removal or capping applied**, consistent with Handbook D.11/D.22 (never remove outliers "solely because values appear extreme") — there's nothing here to justify removing anyway.

## Step 5 — save the cleaned dataset

In [6]:
output_path = PROCESSED_DATA_DIR / "delhi_aqi_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")
print(f"Final shape: {df.shape}")
df.head()

Saved cleaned dataset to: D:\AQI_Forecast_Platform\data\processed\delhi_aqi_cleaned.csv
Final shape: (2191, 9)


,City,Date,AQI,PM2.5,PM10,NO2,SO2,CO,O3
0,Delhi,2018-01-01,406,223.3,438.48,336.98,462.84,4.26,385.7
1,Delhi,2018-01-02,418,229.9,451.44,346.94,476.52,4.39,397.1
2,Delhi,2018-01-03,382,210.1,412.56,317.06,435.48,4.01,362.9
3,Delhi,2018-01-04,366,201.3,395.28,303.78,417.24,3.84,347.7
4,Delhi,2018-01-05,390,214.5,421.20,323.70,444.60,4.10,370.5


## Conclusions

| Step | Finding | Action taken |
|---|---|---|
| Junk columns | 2 fully-empty columns from malformed header | Dropped |
| Duplicate rows | 0 | None needed |
| Missing values | 0 in real columns | None needed |
| Outliers (IQR + Z-score) | 0 on all columns | None removed (detection only, per policy) |

The raw file turned out to need very little correction beyond the header
artifact — the real complexity in this dataset is the **366-day calendar
gap** (found in notebook 01) and the **perfect AQI/pollutant collinearity**
(explored fully in notebook 03), neither of which "cleaning" in the
traditional sense can fix — they're structural characteristics to design
around in feature engineering and modeling.

## Next steps
→ `03_exploratory_data_analysis.ipynb`: full EDA on `data/processed/delhi_aqi_cleaned.csv`.